In [ ]:
# Use %autoreload 1 (explicit reload only) to avoid UnicodeDecodeError on Windows:
# %autoreload 2 reads stdlib sources with cp1252, which fails on UTF-8 files.
# %load_ext autoreload
# %autoreload 1
from napari import Viewer
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# matplotlib.rcParams["font.family"] = "DejaVu Sans"  # or "Arial", "Tahoma", etc.
from napari_pecan_py.widgets import color_tuner

viewer = Viewer()
viewer.window.add_plugin_dock_widget("napari-pecan-py", "Color Tuner")
viewer.window.add_plugin_dock_widget("napari-pecan-py", "Color Adjustments")
# viewer.window.add_plugin_dock_widget("napari-pecan-py", "Mask Retouching")
# viewer.window.add_plugin_dock_widget("napari-pecan-py", "YOLO Segmentation")

# viewer.window.add_plugin_dock_widget("napari-pecan-py", "Contrastive Coding")

(<napari._qt.widgets.qt_viewer_dock_widget.QtViewerDockWidget at 0x1f3263b7a40>,
 <napari_pecan_py.widgets.color_adjustments.widget.ColorAdjustmentsWidget at 0x1f3263b6180>)

We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.
We had to kill ffmpeg to stop it.


In [ ]:
import os
from pathlib import Path
import tempfile

import cv2
import numpy as np
import torch

print("--- CUDA / torch ---")
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device_count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"cuda:{i}:", torch.cuda.get_device_name(i))

# -----------------------------
# Paths (sample_data only)
# -----------------------------
WORKSPACE = Path(r"c:\\Users\\salir\\Desktop\\pecan_py_napari")
DATA_DIR = WORKSPACE / "sample_data"

VIDEO_PATH = DATA_DIR / "GH013558-cropped.MP4"
CRACK_NPY = DATA_DIR / "GH013558-cropped.MP4 - Crack.npy"
KERNEL_NPY = DATA_DIR / "GH013558-cropped.MP4 - Kernel.npy"
PECAN_NPY = DATA_DIR / "GH013558-cropped.MP4 - Pecan.npy"

# -----------------------------
# Small dataset knobs
# -----------------------------
# Keep this small to iterate quickly.
max_frames = 60
imgsz = 256
epochs = 5
batch = 2
lr0 = 1e-3

# -----------------------------
# Load frames from MP4
# -----------------------------
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"Failed to open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
N = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print("--- Video ---")
print(f"name={VIDEO_PATH.name} frames≈{N} size={W}x{H} fps={fps}")

frames = []
while True:
    ok, frame_bgr = cap.read()
    if not ok:
        break
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    frames.append(frame_rgb)
cap.release()

frames = np.stack(frames, axis=0)  # (T,H,W,3)
T = frames.shape[0]
print("Loaded frames:", frames.shape, frames.dtype)

if T > max_frames:
    frames = frames[:max_frames]
T = frames.shape[0]
print("Using T:", T)


# -----------------------------
# Load masks from .npy
# -----------------------------
def normalize_mask(mask: np.ndarray, T: int, H: int, W: int, name: str) -> np.ndarray:
    m = np.asarray(mask)

    # Common shapes we might see: (T,H,W), (H,W), (T,H,W,1)
    if m.ndim == 2:
        m = np.broadcast_to(m[None, ...], (T, H, W))
    elif m.ndim == 3:
        pass
    elif m.ndim == 4:
        # (T,H,W,1) or similar
        m = m[..., 0]
    else:
        raise ValueError(f"Unexpected mask ndim for {name}: {m.shape} (ndim={m.ndim})")

    # Ensure T matches
    if m.shape[0] != T:
        m = m[:T]

    # Ensure H/W match
    if m.shape[1] != H or m.shape[2] != W:
        m2 = np.zeros((T, H, W), dtype=m.dtype)
        for t in range(T):
            m2[t] = cv2.resize(m[t].astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST)
        m = m2

    return m


crack = np.load(CRACK_NPY)
kernel = np.load(KERNEL_NPY)
pecan = np.load(PECAN_NPY)

print("--- Masks (raw) ---")
print("crack:", crack.shape, crack.dtype)
print("kernel:", kernel.shape, kernel.dtype)
print("pecan:", pecan.shape, pecan.dtype)

crack = normalize_mask(crack, T=T, H=H, W=W, name="crack")
kernel = normalize_mask(kernel, T=T, H=H, W=W, name="kernel")
pecan = normalize_mask(pecan, T=T, H=H, W=W, name="pecan")

masks_by_class = {"Crack": crack, "Kernel": kernel, "Pecan": pecan}

# -----------------------------
# Export to YOLO-seg polygon dataset
# -----------------------------
from napari_pecan_py.widgets.yolo_seg.model import export_napari_seg_dataset

with tempfile.TemporaryDirectory(prefix="pecan_yolo_seg_nb_") as tmpdir:
    spec = export_napari_seg_dataset(frames, masks_by_class, tmpdir)

    print("--- Exported YOLO dataset ---")
    print("root:", spec.root)
    print("class_names:", spec.class_names)
    print("data_yaml:", spec.data_yaml)

    # -----------------------------
    # Train (CUDA + logs)
    # -----------------------------
    from ultralytics import YOLO

    device = "0" if torch.cuda.is_available() else "cpu"
    print("--- Training ---")
    print("device:", device)
    print(f"epochs={epochs} batch={batch} lr0={lr0} imgsz={imgsz}")

    model = YOLO("yolov8n-seg.pt")

    # Some ultralytics versions may not support `plots=`.
    try:
        results = model.train(data=str(spec.data_yaml), epochs=epochs, batch=batch, lr0=lr0, device=device, imgsz=imgsz, workers=0, verbose=True, plots=False)
    except TypeError as e:
        print("Retrying training without `plots=` due to TypeError:")
        print(e)
        results = model.train(data=str(spec.data_yaml), epochs=epochs, batch=batch, lr0=lr0, device=device, imgsz=imgsz, workers=0, verbose=True)

    # Ultralytics returns a Results object; best weights should be available on it.
    best = getattr(results, "best", None)
    print("--- Done ---")
    print("best weights:", best)